## Making Pathway Graphs

Inside the paper of this project, we also enriched the top-ranked genes of each topic by treating them as a gene set. Recall that the most biologically significant topics learned were topic 1 of our COVID WAE and topic 6 of our CRC WAE, thus we perform enrichment on these topics. Theoretically, we could perform enrichment on any topic's top gene sets.

In [1]:
import sys
sys.path.append('WTM')

import pickle as p
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

import new_ds
from new_ds import mm_Dataset
from WTM_model import WTM
from utils import calc_topic_uniqueness
import re
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
pd.set_option('display.max_rows', None)
import pickle

import seaborn as sns
import time

In [2]:
def my_softmax(df):
    
    maxes = df.max(axis=1)
    diffs = df.sub(maxes, axis=0)
    num = np.exp(diffs)
    denom = num.sum(axis=1)
    out = num.div(denom, axis=0)
    return out

def load_model(model_prefix, docSet):
    checkpoint = torch.load(f'WTM_checkpoints/{model_prefix}')
    param = checkpoint["param"]
    model = WTM(**param)
    model.load_model(checkpoint["net"])
    model.id2token = {v: k for k,v in docSet.dictionary.token2id.items()}
    return model

def get_embeds(model, docSet, labels_fname = None):
    embeds = model.get_embed(train_data=docSet, num=1000)

    if labels_fname:
        with open(labels_fname, 'r') as file:
            labels_list = [line.strip() for line in file.readlines()]
        out_df = pd.DataFrame(embeds, index=labels_list)
        return out_df
        
    return pd.DataFrame(embeds)

def calculate_phis(docSet, model):
    out_dict = {}
    all_phi_raw = model.get_topic_word_dist(normalize=False)
    headers = np.concatenate([expmat.columns for expmat in docSet.expmats.values()])
    all_phi_raw = pd.DataFrame(all_phi_raw, columns = headers)
    for data_type in np.unique(docSet.label_type):
        type_raw = all_phi_raw.iloc[:, docSet.label_type==data_type]
        out_dict[data_type] = my_softmax(type_raw)
    return out_dict

In [ ]:
ckpts_covid_t10_a5e2 = [f'covid_tp10_a0.05_lr0.0005/{i*10}.ckpt' for i in range(1, 51)]# best one!
ckpts_crc_t10_a5e2 = [f'crc_tp10_a0.05_lr0.0005/{i*10}.ckpt' for i in range(1, 51)]
ckpts_ibd_t10_a5e2 = [f'ibd_tp10_a0.05_lr0.0005/{i*10}.ckpt' for i in range(1, 51)]
ckpts_ibs_t10_a5e2 = [f'ibs_tp10_a0.05_lr0.0005/{i*10}.ckpt' for i in range(1, 51)]

ckpts_crc = [f'crc_model_tp10_a0.1_lr0.001/{i*10}.ckpt' for i in range(1, 11)]
ckpts_ibd = [f'ibd_model_tp10_a0.1_lr0.001/{i*10}.ckpt' for i in range(1, 11)]
ckpts_ibs = [f'ibs_model_tp10_a0.1_lr0.001/{i*10}.ckpt' for i in range(1, 11)]

In [3]:
covid_docSet = mm_Dataset(rebuild=True,
                        data_dir='/gpfs/gibbs/project/gerstein/rtl35/privacy_network/covid_data',
                        mode_names=['gene', 'microbeR', 'premiR'],
                        expmat_fnames=['covid_gene.csv', 'covid_microbeR.csv', 'covid_premiR.csv'],
                        metadata_fname='covid_metadata.csv',
                        out_fname='covid_docDataset.pkl',
                        scale='gene'
                       )

crc_docSet = mm_Dataset(rebuild=True,
                        data_dir = '/gpfs/gibbs/project/gerstein/rtl35/privacy_network/crc_data',
                        mode_names = ['gene', 'microbeR'],
                        expmat_fnames = ['crc_gene.csv', 'crc_microbeR.csv'],
                        metadata_fname = 'crc_metadata.csv',
                        out_fname = 'crc_docDataset.pkl'
                       )

ibd_docSet = mm_Dataset(rebuild=True,
                        data_dir = '/gpfs/gibbs/project/gerstein/rtl35/privacy_network/ibd_data',
                        mode_names = ['gene', 'microbeR'],
                        expmat_fnames = ['ibd_gene.csv', 'ibd_microbeR.csv'],
                        metadata_fname = 'ibd_metadata.csv',
                        out_fname = 'ibd_docDataset.pkl')

ibs_docSet = mm_Dataset(rebuild=True,
                        data_dir = '/gpfs/gibbs/project/gerstein/rtl35/privacy_network/ibs_data',
                        mode_names = ['gene', 'microbeR'],
                        expmat_fnames = ['ibs_gene.csv', 'ibs_microbeR.csv'],
                        metadata_fname = 'ibs_metadata.csv',
                        out_fname = 'ibs_docDataset.pkl')

# ibs_logSet = mm_Dataset(rebuild=True,
#                         data_dir = '/gpfs/gibbs/project/gerstein/rtl35/privacy_network/ibs_data',
#                         mode_names = ['gene', 'microbeR'],
#                         expmat_fnames = ['ibs_gene.csv', 'ibs_microbeR.csv'],
#                         metadata_fname = 'ibs_metadata.csv',
#                         out_fname = 'ibs_docDataset.pkl', log=True, do_normalize=False)

Rebuilding
successfully saved after rebuilding
Rebuilding
successfully saved after rebuilding
Rebuilding
successfully saved after rebuilding
Rebuilding
successfully saved after rebuilding
